# 01 · Tool calling from scratch

**Session 5 — Agent Architecture and Ecosystem Tools · MBRF Data Analytics**

Goal: understand what an *agent* really is by building one with nothing but the model provider's SDK.

1. Call an LLM from Python.
2. Give it a **tool** and see how it asks us to run it.
3. Write the **agent loop**: model → tool → model → … → answer.
4. Add tools of your own and think about how to design them.
5. Put a human in the loop for actions that change things.

The bank in this notebook (*Demo Bank*) and all its data are synthetic.

## 0. Setup

In [ ]:
%pip install -q "openai>=2"

In [ ]:
import json
import os
import sqlite3
import urllib.request
from pathlib import Path

from openai import OpenAI

# Which provider to use. Both are called through the same OpenAI-compatible API.
PROVIDER = "openai"  # "openai" | "gemini"

MODELS = {"openai": "gpt-5.4-mini", "gemini": "gemini-3.5-flash-lite"}
BASE_URLS = {"openai": None, "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/"}
KEY_NAMES = {"openai": "OPENAI_API_KEY", "gemini": "GEMINI_API_KEY"}


def get_api_key(name: str) -> str:
    """Colab secrets (key icon on the left) > environment variable or .env file > ask."""
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except ImportError:
        pass
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    from getpass import getpass

    return getpass(f"{name}: ")


MODEL = MODELS[PROVIDER]
client = OpenAI(api_key=get_api_key(KEY_NAMES[PROVIDER]), base_url=BASE_URLS[PROVIDER], max_retries=5)  # retries 429/5xx with backoff
print("Using", PROVIDER, MODEL)

In [ ]:
# Synthetic bank database used by the tools.
# Synthetic bank database used by the tools: from the repo if cloned, otherwise downloaded (Colab).
DATA_URL = "https://raw.githubusercontent.com/albertotb/curso-agentes-ia/main/data/raw"
DB_PATH = Path("../../data/raw/bank.db")
if not DB_PATH.exists():
    DB_PATH = Path("bank.db")
    if not DB_PATH.exists():
        urllib.request.urlretrieve(f"{DATA_URL}/bank.db", DB_PATH)

db = sqlite3.connect(DB_PATH)
db.row_factory = sqlite3.Row
for table in ["customers", "loans", "transactions", "capital_position"]:
    print(table, db.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0])

## 1. A plain LLM call

A chat model takes a list of **messages** (system, user, assistant…) and returns the next assistant message.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant for a bank's risk department."},
        {"role": "user", "content": "What was Demo Bank's CET1 ratio at 30 September 2026?"},
    ],
)
print(response.choices[0].message.content)

The model cannot know: the figure lives in *our* database. Either it says so, or — worse — it makes up a plausible number.

## 2. Giving the model a tool

A tool is described to the model with a **name**, a **description** and a **JSON Schema** for its parameters. The model never runs anything: it only *asks* us to call the tool with some arguments.

In [ ]:
def get_capital_ratios(reference_date: str) -> dict:
    """Capital and liquidity ratios of Demo Bank at a quarter end, computed from the capital_position table."""
    row = db.execute("SELECT * FROM capital_position WHERE reference_date = ?", (reference_date,)).fetchone()
    if row is None:
        available = [r[0] for r in db.execute("SELECT reference_date FROM capital_position ORDER BY 1")]
        return {"error": f"No data for {reference_date}. Available quarter ends: {available}"}
    rwa = row["rwa_credit_eur_m"] + row["rwa_market_eur_m"] + row["rwa_operational_eur_m"]
    tier1 = row["cet1_capital_eur_m"] + row["at1_capital_eur_m"]
    total = tier1 + row["tier2_capital_eur_m"]
    return {
        "reference_date": reference_date,
        "total_rwa_eur_m": round(rwa, 1),
        "cet1_ratio_pct": round(100 * row["cet1_capital_eur_m"] / rwa, 2),
        "tier1_ratio_pct": round(100 * tier1 / rwa, 2),
        "total_capital_ratio_pct": round(100 * total / rwa, 2),
        "leverage_ratio_pct": round(100 * tier1 / row["leverage_exposure_eur_m"], 2),
        "lcr_pct": round(100 * row["hqla_eur_m"] / row["net_outflows_30d_eur_m"], 1),
    }


get_capital_ratios("2026-09-30")

In [ ]:
capital_tool = {
    "type": "function",
    "function": {
        "name": "get_capital_ratios",
        "description": (
            "Get Demo Bank's regulatory capital and liquidity ratios (CET1, Tier 1, total capital, leverage, LCR) "
            "at a quarter-end date. Use it for any question about the bank's own capital or liquidity figures. "
            "Ratios are already computed: do not recompute them."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "reference_date": {"type": "string", "description": "Quarter-end date in YYYY-MM-DD format, e.g. 2026-09-30."},
            },
            "required": ["reference_date"],
        },
    },
}

messages = [
    {"role": "system", "content": "You are a concise assistant for Demo Bank's risk department."},
    {"role": "user", "content": "What was our CET1 ratio at 30 September 2026?"},
]
response = client.chat.completions.create(model=MODEL, messages=messages, tools=[capital_tool])
msg = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print("content:", msg.content)
print("tool_calls:", msg.tool_calls)

Instead of an answer we get a **tool call**: the name of the function and its arguments as JSON. Now *our code* decides whether to run it, runs it, and sends the result back as a `tool` message.

In [ ]:
messages.append(msg)  # the assistant's tool-call request goes into the history
for call in msg.tool_calls:
    args = json.loads(call.function.arguments)
    result = get_capital_ratios(**args)
    messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

response = client.chat.completions.create(model=MODEL, messages=messages, tools=[capital_tool])
print(response.choices[0].message.content)

## 3. The agent loop

Repeat until the model answers without asking for tools. That loop **is** the agent (the *ReAct* pattern: reason → act → observe).

Note the two safety nets: a maximum number of steps, and tool errors returned to the model as text so it can correct itself.

In [ ]:
TOOLS = {"get_capital_ratios": (get_capital_ratios, capital_tool)}
SYSTEM_PROMPT = "You are an assistant for Demo Bank's risk department. Answer only from tool results; if the tools cannot answer, say so. Be concise and quote figures with their reference date."


def run_agent(question: str, tools: dict = TOOLS, max_steps: int = 8, verbose: bool = True) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    schemas = [schema for _, schema in tools.values()]
    for step in range(max_steps):
        msg = client.chat.completions.create(model=MODEL, messages=messages, tools=schemas).choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for call in msg.tool_calls:
            name, args = call.function.name, json.loads(call.function.arguments or "{}")
            if verbose:
                print(f"[step {step}] {name}({args})")
            try:
                result = tools[name][0](**args)
            except Exception as e:  # unknown tool, wrong arguments, failing query...
                result = {"error": f"{type(e).__name__}: {e}"}
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result, default=str)})
    return "Stopped: too many steps."


print(run_agent("How did our CET1 and leverage ratios evolve over 2026? Is the trend a concern?"))

Things to notice:
- The model may call the tool several times (one per quarter), sometimes **in parallel** in a single step.
- It chose the dates itself from the question.
- It did no arithmetic: the ratios come from Python. **LLMs are bad at arithmetic — put calculations in tools.**

## 4. Exercise: add a tool

Write a tool `get_customer_loans(customer_id)` that returns the customer's segment, internal rating and the list of their loans (product, outstanding amount, IFRS 9 stage, days past due).

1. Write the Python function (query the `customers` and `loans` tables).
2. Write its schema. Think about the **description**: when should the model use it?
3. Register it in `TOOLS` and try the questions below.

Tables: `customers(customer_id, name, segment, country, sector, internal_rating, kyc_risk, onboarding_date)`, `loans(loan_id, customer_id, product, principal_eur, outstanding_eur, interest_rate, start_date, maturity_date, ifrs9_stage, days_past_due, collateral)`.

In [ ]:
def get_customer_loans(customer_id: str) -> dict:
    # TODO
    ...


customer_loans_tool = {
    "type": "function",
    "function": {
        "name": "get_customer_loans",
        "description": "TODO",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
}

# TOOLS["get_customer_loans"] = (get_customer_loans, customer_loans_tool)

In [ ]:
# print(run_agent("Does customer C-0015 have any loan in IFRS 9 stage 2 or 3?"))
# print(run_agent("Summarise the credit exposure of customer C-0013."))
# print(run_agent("What is the CET1 ratio of customer C-0013?"))  # should it call a tool?

## 5. Tool design: a tempting bad tool

Why not give the model a single, all-powerful tool?

In [ ]:
def run_sql(query: str) -> list:
    return [dict(r) for r in db.execute(query).fetchmany(50)]


run_sql_tool = {
    "type": "function",
    "function": {
        "name": "run_sql",
        "description": "Run any SQL query on the bank database.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
    },
}

print(run_agent("Which customers with high KYC risk made cash transactions between 9,000 and 10,000 EUR?", tools={"run_sql": (run_sql, run_sql_tool)}))

It often works — so what is wrong with it?

- **Control:** the model can read *anything*, including data the user should not see, and nothing stops a `DELETE` or `DROP TABLE`.
- **Reliability:** it has to guess table and column names; wrong guesses cost extra steps (watch the errors it gets back).
- **Auditability:** "the agent ran some SQL" is hard to validate; "the agent called `get_customer_loans(C-0042)`" is easy.

Good tools are **narrow, typed and domain-specific**:

| | Bad | Good |
|---|---|---|
| Name | `query_db` | `get_customer_loans` |
| Description | "Runs a query" | What it returns, when to use it, when **not** to |
| Parameters | free text | typed, constrained (`enum`, date formats) |
| Output | 10,000 raw rows | only what is needed, already aggregated |
| Errors | exception | a message the model can act on ("IDs look like C-0001") |
| Side effects | mixed with reads | separate, and approved by a human |

## 6. Human in the loop for actions

Reading data is one thing; **acting** (blocking a card, filing an alert, sending an email) is another. A write tool should not run just because the model asked. In the manual loop that is one `if`.

In [ ]:
REVIEW_QUEUE = []


def flag_customer_for_review(customer_id: str, reason: str) -> dict:
    """Open an AML review case for a customer."""
    answer = input(f"Agent wants to flag {customer_id} for AML review.\nReason: {reason}\nApprove? [y/N] ")
    if answer.strip().lower() != "y":
        return {"status": "rejected by reviewer", "customer_id": customer_id}
    REVIEW_QUEUE.append({"customer_id": customer_id, "reason": reason})
    return {"status": "case opened", "customer_id": customer_id}


flag_tool = {
    "type": "function",
    "function": {
        "name": "flag_customer_for_review",
        "description": "Open an AML review case for a customer. Requires human approval. Use only after checking the customer's transactions.",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "Customer ID, e.g. C-0001"},
                "reason": {"type": "string", "description": "Short factual justification, citing the transactions."},
            },
            "required": ["customer_id", "reason"],
        },
    },
}

aml_tools = {"run_sql": (run_sql, run_sql_tool), "flag_customer_for_review": (flag_customer_for_review, flag_tool)}
print(run_agent("Check customer C-0046's transactions for structuring (cash just under 10,000 EUR) and flag them if needed.", tools=aml_tools))
print(REVIEW_QUEUE)

## Takeaways

- An agent = **LLM + tools + a loop**. About 30 lines of Python.
- The model decides *what* to call; **your code decides whether and how** it runs. That is where controls, permissions and audit logs live.
- Most of an agent's quality comes from **tool design**, not from the prompt.
- Next notebook: the same agent with **LangChain**, and what a framework adds (and does not add).

---
### Solution to exercise 4

In [ ]:
def get_customer_loans(customer_id: str) -> dict:
    customer = db.execute("SELECT customer_id, name, segment, internal_rating FROM customers WHERE customer_id = ?", (customer_id,)).fetchone()
    if customer is None:
        return {"error": f"Customer {customer_id} not found. IDs look like C-0001."}
    loans = db.execute(
        "SELECT loan_id, product, outstanding_eur, ifrs9_stage, days_past_due FROM loans WHERE customer_id = ?",
        (customer_id,),
    ).fetchall()
    return {**dict(customer), "loans": [dict(r) for r in loans]}


customer_loans_tool = {
    "type": "function",
    "function": {
        "name": "get_customer_loans",
        "description": (
            "Get a customer's profile (segment, internal rating 1=best..8=default) and their loans with outstanding "
            "amount, IFRS 9 stage and days past due. Use it for questions about a specific customer's credit exposure."
        ),
        "parameters": {
            "type": "object",
            "properties": {"customer_id": {"type": "string", "description": "Customer ID, e.g. C-0013"}},
            "required": ["customer_id"],
        },
    },
}
TOOLS["get_customer_loans"] = (get_customer_loans, customer_loans_tool)